<a href="https://colab.research.google.com/github/vrmmm/vrmmm/blob/main/Odev3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [66]:
# Gerekli kütüphaneler
!pip install -q sentence-transformers transformers tensorflow scikit-learn

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer, util
from transformers import GPT2Tokenizer, AutoTokenizer, AutoModelForCausalLM
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [67]:
from google.colab import files

df = pd.read_csv("/content/belediye_mesajlari")
departmanlar = sorted(df["departman"].unique())
etiketler = df["departman"].map(
# 5 departman adı
{d: i for i, d in enumerate(departmanlar)}).values # ad -> 0..4

sorted(...): Departman isimlerini alfabetik sıraya koyar (sonuç tutarlı olsun diye).
map: Her departman adını 0,1,2,3,4 numarasına çevirir. Sinir ağı sadece sayı ister.

In [68]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
print("GPT-2 pencere boyutu:", tokenizer.model_max_length)  # genelde 1024

GPT-2 pencere boyutu: 1024


In [69]:
#Token sayılarını hesaplama
token_sayilari = [len(tokenizer.encode(m)) for m in df["mesaj"]]
toplam = sum(token_sayilari)

print("Ortalama token:", np.mean(token_sayilari))
print("En fazla token:", max(token_sayilari))
print("Toplam token:", toplam)
print("Pencereye sığıyor mu?", toplam <= tokenizer.model_max_length)

Ortalama token: 23.18
En fazla token: 37
Toplam token: 2318
Pencereye sığıyor mu? False


tokenizer.encode(m) → mesajı token listesine çevirir.
len(...) → o mesajın kaç token olduğunu verir.
GPT-2’nin bağlam penceresi (model_max_length) genellikle 1024’tür. Toplam token bu sayının altındaysa “sığıyor” deriz.

Türkçe mesajlar GPT-2 tokenizer’ı (İngilizce ağırlıklı BPE) ile daha fazla token’a bölünür çünkü Türkçe ekler, özel karakterler ve sık kullanılmayan kelimeler tek bir token yerine birden fazla alt-token’a ayrılır.

In [70]:
#Embedding ve Anlam Benzerliği
model_emb = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")
X = model_emb.encode(df["mesaj"].tolist(), show_progress_bar=True)
print("X boyutu:", X.shape)   # (100, 768) olmalı

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

X boyutu: (100, 768)


Bu model çok dilli ve 768 boyutlu vektör üretir.
encode tüm mesajları vektöre çevirir → X matrisi (100 satır, 768 sütun).

In [71]:
# Benzerlik Karşılaştırmaları
# Aynı departmandan, ortak kelimesi olmayan iki mesaj seç (kendin bakıp seç)
# Örnek (gerçek indeksleri CSV’ye göre değiştir):
i1, i2 = 0, 5          # aynı departman, kelime örtüşmesi az olanlar
i3, i4 = 0, 25         # farklı departman

sim_ayni = util.cos_sim(X[i1], X[i2]).item()
sim_farkli = util.cos_sim(X[i3], X[i4]).item()

print("Aynı departman benzerliği:", round(sim_ayni, 3))
print("Farklı departman benzerliği:", round(sim_farkli, 3))

Aynı departman benzerliği: 0.604
Farklı departman benzerliği: 0.224


In [72]:
X_egitim, X_test, y_egitim, y_test, m_egitim, m_test = train_test_split(
    X, etiketler, df["mesaj"].values,
    test_size=0.2,
    stratify=etiketler,
    random_state=42
)

print("X_egitim:", X_egitim.shape)   # (80, 768)
print("X_test:", X_test.shape)       # (20, 768)
print("Test etiket dağılımı:", np.bincount(y_test))  # her departmandan 4 olmalı

X_egitim: (80, 768)
X_test: (20, 768)
Test etiket dağılımı: [4 4 4 4 4]


In [73]:
siniflandirici = keras.Sequential([
    layers.Input(shape=(768,)),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(5, activation="softmax")   # 5 departman
])

siniflandirici.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history = siniflandirici.fit(
    X_egitim, y_egitim,
    epochs=30,          # gerekirse artır/azalt
    batch_size=16,
    validation_data=(X_test, y_test),
    verbose=1
)

# Doğruluklar
_, egitim_dogrulugu = siniflandirici.evaluate(X_egitim, y_egitim, verbose=0)
_, test_dogrulugu = siniflandirici.evaluate(X_test, y_test, verbose=0)

print("Eğitim doğruluğu: %", round(egitim_dogrulugu * 100, 1))
print("Test doğruluğu: %", round(test_dogrulugu * 100, 1))

Epoch 1/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 82ms/step - accuracy: 0.3250 - loss: 1.5906 - val_accuracy: 0.4000 - val_loss: 1.5049
Epoch 2/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.5875 - loss: 1.4047 - val_accuracy: 0.6500 - val_loss: 1.3668
Epoch 3/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.7875 - loss: 1.2409 - val_accuracy: 0.7000 - val_loss: 1.1919
Epoch 4/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.8375 - loss: 0.9951 - val_accuracy: 0.7500 - val_loss: 1.0023
Epoch 5/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.8875 - loss: 0.7688 - val_accuracy: 0.7500 - val_loss: 0.8072
Epoch 6/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.9250 - loss: 0.5615 - val_accuracy: 0.8500 - val_loss: 0.6375
Epoch 7/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.9250 - loss: 0.3899 - val_accuracy: 0.8500 - val_loss: 0.5372
Epoch 8/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.9500 - loss: 0.2871 - val_accuracy: 0.8500 - val_loss: 0.5043


Eğitim doğruluğu testten çok yüksekse model ezberliyor (overfitting). Fark makul ise genelliyor demektir. Küçük veri seti olduğu için biraz ezber eğilimi normaldir.

In [74]:
# Yönlendirme ve Güven Eşiği
ESIK = 60  # yüzde

def yonlendir(mesaj):
    vektor = model_emb.encode([mesaj])
    olasiliklar = siniflandirici.predict(vektor, verbose=0)[0]
    en_iyi = np.argmax(olasiliklar)
    guven = round(float(olasiliklar[en_iyi]) * 100)
    if guven < ESIK:
        return "Temsilciye aktar", guven
    return departmanlar[en_iyi], guven

In [75]:
# Yanlış tahmin edilen test mesajları:
tahminler = siniflandirici.predict(X_test, verbose=0)
tahmin_etiket = np.argmax(tahminler, axis=1)

yanlis_indeksler = np.where(tahmin_etiket != y_test)[0]
print("Yanlış sayısı:", len(yanlis_indeksler))

for idx in yanlis_indeksler:
    print("-" * 50)
    print("Mesaj:", m_test[idx])
    print("Gerçek:", departmanlar[y_test[idx]])
    print("Tahmin:", departmanlar[tahmin_etiket[idx]])
    print("Güven:", round(float(np.max(tahminler[idx])) * 100), "%")

Yanlış sayısı: 3
--------------------------------------------------
Mesaj: Terk edilmiş bir araç aylardır sokakta duruyor.
Gerçek: Zabıta
Tahmin: Temizlik ve Çöp
Güven: 74 %
--------------------------------------------------
Mesaj: Gece yarısı inşaat çalışması yapılıyor, uyuyamıyoruz.
Gerçek: Zabıta
Tahmin: Ulaşım ve Trafik
Güven: 72 %
--------------------------------------------------
Mesaj: Lağım kokusu bütün sokağı sardı.
Gerçek: Su ve Kanalizasyon
Tahmin: Temizlik ve Çöp
Güven: 99 %


In [76]:
# 5 yeni mesaj
yeni_mesajlar = [
    "Sokaktaki çöp konteyneri taşmış, koku çok kötü.",
    "Ana cadde de trafik ışığı çalışmıyor.",
    "Parktaki çocuk oyun grubu kırılmış.",
    "Su borusu patlamış, yol tamamen su altında kalmış.",   # belirsiz (Su + Park/Ulaşım)
    "Gece gürültü yapan dükkân var, zabıta gelsin.",
    # ... en az 5 tane yaz
]

for m in yeni_mesajlar:
    dept, guven = yonlendir(m)
    print(f"{m[:60]}... → {dept} ({guven}%)")

Sokaktaki çöp konteyneri taşmış, koku çok kötü.... → Temizlik ve Çöp (100%)
Ana cadde de trafik ışığı çalışmıyor.... → Ulaşım ve Trafik (100%)
Parktaki çocuk oyun grubu kırılmış.... → Park ve Bahçeler (100%)
Su borusu patlamış, yol tamamen su altında kalmış.... → Su ve Kanalizasyon (98%)
Gece gürültü yapan dükkân var, zabıta gelsin.... → Zabıta (100%)


In [77]:
# sohbet modeli
!pip install -q transformers accelerate sentencepiece

In [78]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_adi = "Qwen/Qwen2.5-0.5B-Instruct"

sohbet_tok = AutoTokenizer.from_pretrained(model_adi)
sohbet_model = AutoModelForCausalLM.from_pretrained(
    model_adi,
    torch_dtype=torch.float16,          # bellek tasarrufu
    device_map="auto"                  # GPU varsa otomatik kullanır
)

print("Model yüklendi!")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model yüklendi!


In [79]:
def llm_yonlendir(mesaj):
    talimat = (
        "Bu belediye mesajını şu departmanlardan birine ata: "
        + ", ".join(departmanlar)
        + ". Yalnızca departman adını yaz.\nMesaj: " + mesaj
    )

    mesajlar = [{"role": "user", "content": talimat}]

    metin = sohbet_tok.apply_chat_template(
        mesajlar,
        tokenize=False,
        add_generation_prompt=True
    )

    girdi = sohbet_tok(metin, return_tensors="pt").to(sohbet_model.device)

    with torch.no_grad():
        uretilen = sohbet_model.generate(
            **girdi,
            max_new_tokens=20,
            do_sample=False
        )

    # Sadece yeni üretilen kısmı al
    yeni = uretilen[0][girdi["input_ids"].shape[1]:]
    cevap = sohbet_tok.decode(yeni, skip_special_tokens=True).strip()
    return cevap

In [80]:
# TEST
ornek = "Parktaki salıncak kırılmış, çocuklar düşebilir."
print(llm_yonlendir(ornek))

Departman: Park ve Bahçeler
